# Book places review

Check what Claude read in each book against the scans. Pick a book: you get its places
(grouped spellings, matched modern location) and, for each place, every mention with the
transcribed quote and a link that opens **that page** of the scan on Commons.

Mark each place **right / wrong / unsure** and add a note: your verdicts are saved to
`book_review.csv` in your work folder (yours only; nothing is written to the database),
so you can stop and continue later. The summary cell at the end shows accuracy so far.

Data: `core.book_read.places` and `core.book_page` (the `read_books` DAG, `birddog/books.py`).

In [ ]:
import os, re, html
import pandas as pd
import psycopg
import ipywidgets as w
from IPython.display import display, HTML

DB_URL = os.environ["BIRDDOG_DATABASE_URL"]
REVIEW_FILE = os.path.expanduser("~/work/book_review.csv")

with psycopg.connect(DB_URL, autocommit=True) as conn:
    books = pd.read_sql("""
        SELECT b.document_id, d.title, d.url, b.page_count, b.cost_usd::float AS cost_usd, b.places
        FROM core.book_read b JOIN core.document d ON d.id = b.document_id
        WHERE b.status = 'done' ORDER BY b.document_id""", conn)
    pages = pd.read_sql("""
        SELECT DISTINCT ON (document_id, page_no) document_id, page_no, model, result
        FROM core.book_page
        ORDER BY document_id, page_no, (model = 'claude-opus-5-5') DESC, created_at DESC""", conn)

def page_link(url, page_no):
    m = re.search(r"(File:[^?#]+)$", url)
    return f"https://commons.wikimedia.org/w/index.php?title={m.group(1)}&page={page_no}" if m else url

mentions = []
for r in pages.itertuples():
    for p in (r.result or {}).get("places", []):
        mentions.append({"document_id": r.document_id, "page_no": r.page_no, "model": r.model, **p})
mentions = pd.DataFrame(mentions)

if os.path.exists(REVIEW_FILE):
    reviews = pd.read_csv(REVIEW_FILE)
else:
    reviews = pd.DataFrame(columns=["document_id", "place_no", "name_en", "verdict", "note"])
print(f"{len(books)} books ready for review, {len(mentions)} mentions; {len(reviews)} verdicts saved so far")

## Review a book

In [ ]:
book_pick = w.Dropdown(
    options=[(f"{r.document_id} · {r.page_count} p. · {r.title[:70]}", r.document_id) for r in books.itertuples()],
    description="Book", layout=w.Layout(width="95%"))
out = w.Output()

def verdict_of(doc_id, place_no):
    hit = reviews[(reviews["document_id"] == doc_id) & (reviews["place_no"] == place_no)]
    return (hit.iloc[-1]["verdict"], hit.iloc[-1]["note"] if isinstance(hit.iloc[-1]["note"], str) else "") if len(hit) else ("unsure", "")

def save(doc_id, place_no, name_en, verdict, note):
    global reviews
    reviews = reviews[~((reviews["document_id"] == doc_id) & (reviews["place_no"] == place_no))]
    reviews = pd.concat([reviews, pd.DataFrame([{"document_id": doc_id, "place_no": place_no, "name_en": name_en,
                                                 "verdict": verdict, "note": note}])], ignore_index=True)
    reviews.to_csv(REVIEW_FILE, index=False)

def show(doc_id):
    book = books[books["document_id"] == doc_id].iloc[0]
    places = (book["places"] or {}).get("places", [])
    summary = (book["places"] or {}).get("summary", {})
    rows = [w.HTML(f'<h3 style="margin:4px 0">{html.escape(book["title"])}</h3>'
                   f'<a href="{book["url"]}" target="_blank" rel="noopener">open the scan on Commons</a> · '
                   f'{book["page_count"]} pages · ${book["cost_usd"]:.2f} · {summary.get("places", 0)} places, '
                   f'{summary.get("matched", 0)} matched, illegible pages: {summary.get("illegible_pages") or "none"}')]
    book_mentions = mentions[mentions["document_id"] == doc_id]
    for i, place in enumerate(places, start=1):
        variants = set(place.get("variants", []))
        quotes = book_mentions[book_mentions["as_written"].isin(variants)].sort_values("page_no")
        links = " ".join(
            f'<a href="{page_link(book["url"], q.page_no)}" target="_blank" rel="noopener" '
            f'title="{html.escape(str(q.quote))}">p.{q.page_no}</a>' for q in quotes.drop_duplicates("page_no").itertuples())
        sample = "<br>".join(
            f'p.{q.page_no}: «{html.escape(str(q.quote))}» <span style="color:#52514e">({q.confidence}, {q.model.split("-")[1]})</span>'
            for q in quotes.head(4).itertuples())
        match = (f'→ <b>{html.escape(str(place.get("location_name")))}</b> (core.location {place.get("location_id")}, '
                 f'score {place.get("match_score")})') if place.get("location_id") else '→ <i>no match in core.location</i>'
        head = w.HTML(
            f'<div style="border-top:1px solid #e8e7e3;padding-top:6px"><b>{i}. {html.escape(place.get("name_uk",""))}'
            f' / {html.escape(place.get("name_en",""))}</b> · {place.get("mentions")} mentions · '
            f'{html.escape(place.get("admin_context") or "")} {match}<br>'
            f'<span style="color:#52514e">spellings: {html.escape(" | ".join(sorted(variants)))}</span><br>'
            f'{sample}<br>pages: {links}</div>')
        current, note_text = verdict_of(doc_id, i)
        pick = w.ToggleButtons(options=["right", "wrong", "unsure"], value=current, style={"button_width": "70px"})
        note = w.Text(value=note_text, placeholder="note (e.g. 'stamp, not a place', 'should be Koroshov')",
                      layout=w.Layout(width="420px"))
        def on_change(_, i=i, place=place, pick=pick, note=note):
            save(doc_id, i, place.get("name_en", ""), pick.value, note.value)
        pick.observe(on_change, names="value")
        note.observe(on_change, names="value")
        rows.append(w.VBox([head, w.HBox([pick, note])]))
    with out:
        out.clear_output()
        display(w.VBox(rows))

book_pick.observe(lambda change: show(change["new"]), names="value")
display(book_pick, out)
if len(books):
    show(book_pick.value)

## Accuracy so far

In [ ]:
reviews = pd.read_csv(REVIEW_FILE) if os.path.exists(REVIEW_FILE) else reviews
decided = reviews[reviews["verdict"].isin(["right", "wrong"])]
if decided.empty:
    print("No verdicts yet: mark places right or wrong above.")
else:
    print(f"{len(decided)} places judged: {(decided['verdict'] == 'right').mean():.0%} right")
    display(reviews.groupby(["document_id", "verdict"]).size().unstack(fill_value=0))
    display(reviews[reviews["verdict"] == "wrong"][["document_id", "place_no", "name_en", "note"]])